# Repair Priority
Rank detected damage using its type, relative size and count.


In [ ]:
from pathlib import Path
import sys

# Resolve shared modules and portable project settings.
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/project.json').is_file()),None)
if ROOT is None:
    raise FileNotFoundError('RoadScan project configuration was not found.')
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0,str(ROOT/'src'))
from roadscan.settings import load_settings
settings=load_settings(ROOT)


In [ ]:
import copy
import json
import numpy as np
import pandas as pd
from roadscan.evaluation import load_records
from roadscan.priority import PRIORITY_CONFIG,priority_score

# Apply the same policy to predicted boxes and annotation boxes.
out=Path(settings['output_dir'])
frozen=json.loads((out/'frozen_selection.json').read_text(encoding='utf-8'))
thresholds=np.asarray(frozen['thresholds']);items=[];rows=[]
for split in ['test','saudi_test']:
    for record in load_records(out/f'{split}_predictions.json'):
        pred=record['pred'];pred=pred[pred[:,4]>=thresholds[pred[:,5].astype(int)]]
        truth=record['truth']
        reference=np.column_stack([truth[:,1:],np.ones(len(truth)),truth[:,0]])
        p=priority_score(pred,record['width'],record['height'])
        t=priority_score(reference,record['width'],record['height'])
        items.append((split,pred,record['width'],record['height']))
        rows.append(dict(split=split,image=record['image'],priority=p['score'],band=p['band'],
            truth_priority=t['score'],truth_band=t['band'],needs_review=p['needs_review']))
frame=pd.DataFrame(rows);frame.to_csv(out/'priority_scores.csv',index=False)
display(frame.groupby('split')[['priority','truth_priority']].mean())

# Test ranking stability while keeping box geometry and band boundaries fixed.
rng=np.random.default_rng(42);trials=[]
for trial in range(200):
    cfg=copy.deepcopy(PRIORITY_CONFIG)
    for key in cfg['type_weight']:
        cfg['type_weight'][key]=float(np.clip(cfg['type_weight'][key]*rng.uniform(.75,1.25),.05,1))
    a,b=cfg['weights']['severity']*rng.uniform(.75,1.25),cfg['weights']['count']*rng.uniform(.75,1.25)
    cfg['weights']['severity'],cfg['weights']['count']=float(a/(a+b)),float(b/(a+b))
    altered=[priority_score(p,w,h,cfg) for _,p,w,h in items]
    for split in ['test','saudi_test']:
        mask=frame.split.eq(split).to_numpy();base=frame.loc[mask,'priority'].rank()
        scores=pd.Series([p['score'] for p,m in zip(altered,mask) if m]).rank()
        correlation=base.reset_index(drop=True).corr(scores)
        bands=[p['band'] for p,m in zip(altered,mask) if m]
        trials.append(dict(trial=trial,split=split,spearman=correlation,
            same_band=float(np.mean(frame.loc[mask,'band'].to_numpy()==bands))))
pd.DataFrame(trials).to_csv(out/'priority_sensitivity_trials.csv',index=False)
display(pd.DataFrame(trials).groupby('split')[['spearman','same_band']].mean())
